In [50]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Input

In [21]:
df = pd.read_csv('Churn_Modelling.csv')

In [22]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [23]:
df.shape

(10000, 14)

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


In [25]:
df.duplicated().sum()

np.int64(0)

In [26]:
# Check Imbalnce
df['Exited'].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [27]:
df['Gender'].value_counts()

Gender
Male      5457
Female    4543
Name: count, dtype: int64

In [28]:
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname']) #<-- Dropping these columns for simplyifing the task for now , assumming surname is not important

In [29]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [30]:
df = pd.get_dummies(df,columns=['Gender','Geography'],drop_first=True)
# Basically simply converting classess of each into their own column

In [31]:
df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Gender_Male,Geography_Germany,Geography_Spain
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,False,False,True
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,0,False,False,True


In [33]:
# It is advised that we scale the values before feeding the model so that the model converges faster

In [34]:
X,y = df.drop(columns=['Exited']),df['Exited']

In [35]:
X_train ,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [36]:
X_train.shape,y_train.shape

((8000, 11), (8000,))

In [37]:
X_test.shape,y_test.shape

((2000, 11), (2000,))

In [38]:
# Now scale these value

In [41]:
scaler = StandardScaler()

scaler.fit(X_train)

X_train_scalled = scaler.transform(X_train)
X_test_scalled = scaler.transform(X_test)

In [44]:
X_train_scalled[:3]

array([[ 0.35649971, -0.6557859 ,  0.34567966, -1.21847056,  0.80843615,
         0.64920267,  0.97481699,  1.36766974,  0.91324755, -0.57946723,
        -0.57638802],
       [-0.20389777,  0.29493847, -0.3483691 ,  0.69683765,  0.80843615,
         0.64920267,  0.97481699,  1.6612541 ,  0.91324755,  1.72572313,
        -0.57638802],
       [-0.96147213, -1.41636539, -0.69539349,  0.61862909, -0.91668767,
         0.64920267, -1.02583358, -0.25280688,  0.91324755, -0.57946723,
         1.73494238]])

In [45]:
X_test_scalled[:3]

array([[-0.57749609, -0.6557859 , -0.69539349,  0.32993735,  0.80843615,
        -1.54035103, -1.02583358, -1.01960511,  0.91324755,  1.72572313,
        -0.57638802],
       [-0.29729735,  0.3900109 , -1.38944225, -1.21847056,  0.80843615,
         0.64920267,  0.97481699,  0.79888291,  0.91324755, -0.57946723,
        -0.57638802],
       [-0.52560743,  0.48508334, -0.3483691 , -1.21847056,  0.80843615,
         0.64920267, -1.02583358, -0.72797953, -1.09499335, -0.57946723,
         1.73494238]])

In [47]:
# Import all the libraries as stated at the start

In [51]:
model = Sequential()

model.add(Input(shape=(11,)))

model.add(Dense(3,activation='sigmoid'))

model.add(Dense(1,activation='sigmoid'))

In [52]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 3)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40 (160.00 B)

 Trainable params: 40 (160.00 B)

 Non-trainable params: 0 (0.00 B)